# Подготовка данных

Цель этого ноутбука — сформировать датасет для прогнозирования задержек доставки и подготовить признаки для первой модели.

Прогноз будет выполняться в момент оформления заказа. Поэтому в модели можно использовать только информацию, доступную к этому моменту: характеристики покупателя, товаров, продавцов и состава заказа. Фактические логистические даты и отзывы использовать нельзя, так как они становятся известны позже и приведут к утечке данных.

In [1]:
import pandas as pd
import numpy as np

In [2]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                                                                         'order_delivered_customer_date', 'order_estimated_delivery_date'])
print(orders.shape)

(99441, 8)


In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## Формирование обучающей выборки и целевой переменной

Для обучения оставлены только заказы со статусом `delivered`, поскольку только для них известен фактический результат доставки. Заказы с пропуском фактической даты доставки исключены.

Целевая переменная `is_delayed` показывает, был ли заказ доставлен позже обещанной календарной даты. Доставка в любой момент обещанного дня считается выполненной вовремя.

In [4]:
delivered_orders = orders[orders['order_status'] == 'delivered'].dropna(subset='order_delivered_customer_date')
print(delivered_orders.shape)
print(delivered_orders.isna().sum())

(96470, 8)
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64


In [5]:
delivered_orders['is_delayed'] = delivered_orders['order_delivered_customer_date'].dt.date > delivered_orders['order_estimated_delivery_date'].dt.date

In [6]:
delivered_orders['is_delayed'].value_counts()

is_delayed
False    89936
True      6534
Name: count, dtype: int64

In [7]:
delivered_orders['is_delayed'].value_counts(normalize=True)

is_delayed
False    0.932269
True     0.067731
Name: proportion, dtype: float64

После отбора в выборке осталось 96 470 заказов: 89 936 были доставлены вовремя и 6 534 — с задержкой. Доля задержанных заказов составляет около 6,8%.

Целевая переменная несбалансирована, поэтому при оценке моделей нельзя ориентироваться только на accuracy. Позже потребуются метрики, учитывающие качество определения редкого положительного класса.

## Выбор признаков и предотвращение утечки данных

Прогноз строится в момент оформления заказа. Поэтому в модель могут входить только данные, которые доступны к этому моменту: дата покупки, обещанная дата доставки, характеристики покупателя, состав заказа, характеристики товаров и продавцов.

Идентификаторы `order_id`, `customer_id`, `customer_unique_id`, `product_id` и `seller_id` используются как технические ключи для объединения таблиц, но не будут передаваться в модель как признаки.

Из `orders` не будут использоваться `order_status`, `order_approved_at`, `order_delivered_carrier_date` и `order_delivered_customer_date`. Эти столбцы либо не несут информации после фильтрации, либо становятся известны уже после оформления заказа. Использование фактической даты доставки привело бы к утечке целевой переменной. По той же причине не используются данные отзывов.

## Присоединение информации о покупателях

К отобранным заказам была присоединена таблица `customers` по ключу `customer_id` с помощью левого соединения. После объединения число строк и число уникальных `order_id` не изменились и остались равны 96 470, поэтому каждому заказу соответствует одна строка.

Штат покупателя может быть полезным признаком, так как география влияет на логистику и сроки доставки. Город и ZIP-префикс пока сохраняются в датасете, но не включаются в первую версию модели: они содержат большое число уникальных значений и потребуют отдельной обработки.

In [21]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders_customers = pd.merge(delivered_orders, customers, how='left', on='customer_id')
print(orders_customers['order_id'].nunique())

96470


In [9]:
orders_customers.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delayed,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


## Формирование признаков состава заказа

Таблица `order_items` содержит отдельную строку для каждой позиции заказа. Чтобы получить характеристики товаров и продавцов, к ней присоединяются таблицы `products` и `sellers` по ключам `product_id` и `seller_id`.

До агрегации рассчитывается объём каждой позиции как произведение длины, высоты и ширины товара. Затем расширенная таблица агрегируется по `order_id`, чтобы сформировать признаки уровня заказа.

In [10]:
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
print(order_items.shape)
print(order_items['order_id'].nunique())

(112650, 7)
98666


In [11]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [12]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
print(products.columns)
#т к продукт не может весить 0 граммов, считаем вес за пропуск. В прошлом ноутбуке 0 был только в весе, поэтому габариты мы не трогаем
products['product_weight_g'] = products['product_weight_g'].replace(0, np.nan)
#в названиях колонок были опечатки. Исправлю чтобы потом не путаться
products.rename(columns=
                {'product_name_lenght' : 'product_name_length', 
                 'product_description_lenght' : 'product_description_length',
                }, inplace=True)
items_products = pd.merge(order_items, products, how='left', on='product_id')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
items_products_sellers = pd.merge(items_products, sellers, how='left', on='seller_id')
items_products_sellers.shape[0] == order_items.shape[0]

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')


True

In [13]:
#вместо отдельных длины, высоты и ширины, создадим общий параметр объема
items_products_sellers['product_volume_cm3'] = (
    items_products_sellers['product_length_cm']
    * items_products_sellers['product_height_cm']
    * items_products_sellers['product_width_cm']
)

In [14]:
items_products_sellers.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_volume_cm3
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,27277,volta redonda,SP,3528.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,3471,sao paulo,SP,60000.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,37564,borda da mata,MG,14157.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,14403,franca,SP,2400.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,87900,loanda,PR,42000.0


In [15]:
items_product_sellers_by_order = items_products_sellers.groupby('order_id').agg(
    items=('order_item_id', 'size'), 
    products=('product_id', 'nunique'), 
    sellers=('seller_id', 'nunique'),
    total_price=('price', 'sum'), 
    total_freight_value=('freight_value', 'sum'), 
    categories=('product_category_name', 'nunique'),
    avg_name_len=('product_name_length', 'mean'),
    avg_desc_len=('product_description_length', 'mean'),
    avg_photos_per_item=('product_photos_qty', 'mean'),
    total_weight=('product_weight_g', lambda x: x.sum(min_count=1)),
    total_volume=('product_volume_cm3', lambda x: x.sum(min_count=1)),
    max_volume=('product_volume_cm3', 'max'),
    seller_states=('seller_state', 'nunique')).reset_index()

In [16]:
items_product_sellers_by_order.shape[0] == order_items['order_id'].nunique()

True

Для каждого заказа рассчитаны количество позиций, уникальных товаров, продавцов, категорий и штатов продавцов; суммарная стоимость товаров и доставки; средние характеристики карточек товаров; общий вес и объём заказа, а также максимальный объём одной позиции.

При суммировании веса и объёма полностью отсутствующие значения сохраняются как пропуски, а не превращаются в физически невозможные нули. Их обработка будет выполняться позднее вместе с остальными пропусками.

In [17]:
dataset = pd.merge(orders_customers, items_product_sellers_by_order, how='left', on='order_id')

In [18]:
dataset.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delayed,customer_unique_id,...,total_price,total_freight_value,categories,avg_name_len,avg_desc_len,avg_photos_per_item,total_weight,total_volume,max_volume,seller_states
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,7c396fd4830fd04220f754e42b4e5bff,...,29.99,8.72,1,40.0,268.0,4.0,500.0,1976.0,1976.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,af07308b275d755c9edb36a90c618231,...,118.70,22.76,1,29.0,178.0,1.0,400.0,4693.0,4693.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,3a653a41f6f9fc3d2a113cf8398680e8,...,159.90,19.22,1,46.0,232.0,1.0,420.0,9576.0,9576.0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,7c142cf63193a1473d2e66489a9ae977,...,45.00,27.20,1,59.0,468.0,3.0,450.0,6000.0,6000.0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,72632f0f9dd73dfee390c9b22eb56dd6,...,19.90,8.72,1,38.0,316.0,4.0,250.0,11475.0,11475.0,1


In [19]:
dataset.shape[0] == orders_customers.shape[0]

True

In [22]:
dataset.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
is_delayed                          0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
items                               0
products                            0
sellers                             0
total_price                         0
total_freight_value                 0
categories                          0
avg_name_len                     1332
avg_desc_len                     1332
avg_photos_per_item              1332
total_weight                       22
total_volume                       16
max_volume                         16
seller_states                       0
dtype: int64

In [28]:
dataset['order_purchase_month'] = dataset['order_purchase_timestamp'].dt.month
dataset['order_purchase_dayofweek'] = dataset['order_purchase_timestamp'].dt.dayofweek
dataset['order_purchase_hour'] = dataset['order_purchase_timestamp'].dt.hour
dataset['estimated_delivery_days'] = (dataset['order_estimated_delivery_date'] - dataset['order_purchase_timestamp']) / pd.Timedelta(days=1)

In [30]:
dataset[['order_purchase_month', 'order_purchase_dayofweek', 'order_purchase_hour', 'estimated_delivery_days']].describe()

,order_purchase_month,order_purchase_dayofweek,order_purchase_hour,estimated_delivery_days
count,96470.000000,96470.000000,96470.000000,96470.000000
mean,6.031046,2.756494,14.773028,23.736343
std,3.228479,1.967041,5.328347,8.761052
min,1.000000,0.000000,0.000000,2.008009
25%,3.000000,1.000000,11.000000,18.329905
50%,6.000000,3.000000,15.000000,23.230880
75%,8.000000,4.000000,19.000000,28.407795
max,12.000000,6.000000,23.000000,155.135463


In [31]:
dataset.to_csv('../data/processed/dataset.csv', index=False)

## Итоги подготовки данных

Сформирован датасет из 96 470 доставленных заказов, где одна строка соответствует одному заказу. Для каждого заказа создана целевая переменная `is_delayed`: доля задержанных заказов составляет около 6,8%, поэтому при дальнейшем моделировании необходимо учитывать дисбаланс классов.

К данным о заказах добавлены характеристики покупателей, состава заказа, товаров и продавцов. Также созданы временные признаки: месяц, день недели и час оформления заказа, а также обещанный срок доставки в днях.

После объединений количество строк и уникальных заказов не изменилось. Обнаруженные пропуски в характеристиках товаров сохранены для последующей обработки после разделения данных на обучающую и тестовую выборки.

Фактические логистические даты, идентификаторы и другие служебные столбцы сохранены в подготовленном датасете, но не будут использоваться как признаки модели. Следующий этап — исследовательский анализ собранных признаков и их связи с задержками доставки.
